# 🤖 Building an AI Agent: E-Commerce Customer Support

## What You'll Learn

This tutorial walks you through building a **production-style AI agent** using a realistic business case.
By the end, you'll understand:

1. **What AI agents are** and how they differ from simple chatbots
2. **The ReAct pattern** — the standard architecture for tool-using agents
3. **Tool design** — how to give agents access to external systems
4. **Business rules** — enforcing guardrails inside tools, not the LLM
5. **Measuring success** — KPIs for evaluating agent performance

---

## The Business Problem

**ShopSmart Inc.** processes 50,000 orders/month. Their support team:
- Handles 15,000 tickets/month
- 70% are Tier-1 (order status, refunds, product questions)
- Average response time: 4 hours
- Cost: $12.50 per ticket

**Goal**: Build an AI agent to handle Tier-1 support, reducing costs by 60% and response time to <10 seconds.

---

## Part 1: Understanding AI Agents

### Chatbot vs. Agent

| | Chatbot | AI Agent |
|---|---|---|
| **Input/Output** | Text → Text | Text → Actions + Text |
| **Tools** | None | Can call functions/APIs |
| **Reasoning** | Single pass | Multi-step loop |
| **State** | Stateless or simple | Maintains context |
| **Example** | FAQ bot | Customer support agent |

### The ReAct Pattern

Our agent follows the **ReAct** (Reasoning + Acting) pattern:

```
Customer: "Where is my order #1001?"
        ↓
   ┌─────────┐
   │  THINK  │  → "Customer wants order status. I should look it up."
   └────┬────┘
        ↓
   ┌─────────┐
   │   ACT   │  → Call lookup_order(order_id="1001")
   └────┬────┘
        ↓
   ┌─────────┐
   │ OBSERVE │  → {status: "shipped", tracking: "FX-789...", eta: "Mar 12"}
   └────┬────┘
        ↓
   ┌─────────┐
   │ RESPOND │  → "Your order #1001 has shipped! Tracking: FX-789..."
   └─────────┘
```

The key insight: **the LLM decides WHICH tool to call and WHAT arguments to pass**.

---

## Part 2: Setting Up the Environment

In [ ]:
# Add the project root to the Python path
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Verify imports work
from src.mock_data import CUSTOMERS, PRODUCTS, ORDERS
from src.tools import call_tool, TOOL_REGISTRY
from src.prompts import SYSTEM_PROMPT

print(f"✅ Loaded {len(CUSTOMERS)} customers, {len(PRODUCTS)} products, {len(ORDERS)} orders")
print(f"✅ Registered {len(TOOL_REGISTRY)} tools: {', '.join(TOOL_REGISTRY.keys())}")

---

## Part 3: Exploring the Data Layer

In a real system, this would be a database. Here we use Python dicts to keep things simple.

In [ ]:
# Let's explore our mock e-commerce data

print("=" * 60)
print("CUSTOMERS")
print("=" * 60)
for cid, customer in CUSTOMERS.items():
    print(f"  {cid}: {customer['name']} ({customer['tier']} tier, joined {customer['joined']})")

print(f"\n{'=' * 60}")
print("PRODUCTS")
print("=" * 60)
for pid, product in PRODUCTS.items():
    stock = '✅' if product['in_stock'] else '❌'
    print(f"  {pid}: {product['name']} — ${product['price']:.2f} | ⭐{product['rating']} | {stock}")

print(f"\n{'=' * 60}")
print("ORDERS (first 5)")
print("=" * 60)
for oid, order in list(ORDERS.items())[:5]:
    items = ', '.join(item['name'] for item in order['items'])
    print(f"  #{oid}: {order['status'].upper()} — {items} — ${order['total']:.2f}")

---

## Part 4: Building Agent Tools 🔧

Tools are **functions** the agent can call. Each tool:
- Has a clear **name** and **description** (so the LLM knows when to use it)
- Accepts **structured inputs** (validated parameters)
- Returns **structured outputs** (JSON-like dicts)
- Enforces **business rules** (not the LLM's job!)

Let's test each tool individually.

### Tool 1: `lookup_order`
Retrieves order details by ID.

In [ ]:
import json

# Test: Look up a shipped order
result = call_tool("lookup_order", {"order_id": "1001"})
print("📦 Order #1001 (shipped):")
print(json.dumps(result, indent=2))

print("\n" + "-" * 40 + "\n")

# Test: Look up a non-existent order
result = call_tool("lookup_order", {"order_id": "9999"})
print("❌ Order #9999 (not found):")
print(json.dumps(result, indent=2))

### Tool 2: `process_refund`
Processes a refund **only if business rules are satisfied**.

In [ ]:
# Test: Refund a delivered order (within 30 days) — should SUCCEED
result = call_tool("process_refund", {"order_id": "1002", "reason": "not_satisfied"})
print("✅ Refund for delivered order #1002:")
print(json.dumps(result, indent=2))

print("\n" + "-" * 40 + "\n")

# Test: Refund a shipped order (not delivered yet) — should FAIL
result = call_tool("process_refund", {"order_id": "1001", "reason": "changed_mind"})
print("❌ Refund for shipped order #1001 (not delivered):")
print(json.dumps(result, indent=2))

print("\n" + "-" * 40 + "\n")

# Test: Refund an old order (>30 days) — should FAIL
result = call_tool("process_refund", {"order_id": "1004", "reason": "defective"})
print("❌ Refund for old order #1004 (>30 days):")
print(json.dumps(result, indent=2))

### 💡 Key Insight: Business Rules in Tools, Not the LLM

Notice that the refund tool **enforces its own rules**:
- 30-day return window
- Order must be "delivered" status

We do NOT rely on the LLM to enforce these rules. Why?
1. **LLMs hallucinate** — they might approve an ineligible refund
2. **Rules change** — update the tool, not the prompt
3. **Audit trail** — tool results are deterministic and logged

### Tool 3: `search_products`
Searches the product catalog with keyword and category filters.

In [ ]:
# Test: Search for laptops
result = call_tool("search_products", {"query": "laptop"})
print(f"🛍️ Search 'laptop' — found {result['count']} results:")
for p in result['products']:
    print(f"  • {p['name']} — {p['price']} | ⭐ {p['rating']}")

print("\n" + "-" * 40 + "\n")

# Test: Search by category
result = call_tool("search_products", {"category": "Furniture"})
print(f"🪑 Category 'Furniture' — found {result['count']} results:")
for p in result['products']:
    print(f"  • {p['name']} — {p['price']} | ⭐ {p['rating']}")

### Tool 4: `escalate_to_human`
Creates an escalation ticket for complex issues.

In [ ]:
# Test: Escalate a conversation
result = call_tool("escalate_to_human", {
    "customer_id": "C001",
    "reason": "Customer reports potential fraud on their account",
    "priority": "urgent",
})
print("🚨 Escalation created:")
print(json.dumps(result, indent=2))

---

## Part 5: The Agent Loop (ReAct) 🔄

Now let's put it all together. The agent:
1. Receives a customer message
2. **Thinks** about what the customer needs
3. **Acts** by calling the appropriate tool
4. **Observes** the tool result
5. **Responds** to the customer

Let's run the agent on our three demo scenarios.

In [ ]:
from src.agent import CustomerSupportAgent

# Create the agent (mock mode — no API key needed)
agent = CustomerSupportAgent(use_openai=False)

print("Agent created in MOCK mode (rule-based).")
print("Set OPENAI_API_KEY and use_openai=True for GPT-powered responses.")

### Scenario 1: Order Status Inquiry 📦

Watch the agent's internal reasoning (💭 Think, 👁 Observe) as it processes the request:

In [ ]:
# Scenario 1: Customer asks about order status
response = agent.chat("Hi, can you tell me where my order #1001 is?")

### Scenario 2: Refund Request 💰

This scenario requires TWO tool calls: first look up the order, then process the refund.

In [ ]:
# Reset conversation for a new scenario
from src.prompts import SYSTEM_PROMPT
agent.conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]

# Scenario 2: Customer wants a refund
response = agent.chat("I'd like to get a refund for order #1002. The headphones aren't what I expected.")

### Scenario 3: Product Recommendations 🛍️

In [ ]:
# Reset conversation
agent.conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]

# Scenario 3: Customer wants product recommendations
response = agent.chat("Can you suggest a good laptop for coding?")

---

## Part 6: How the OpenAI Integration Works

When using OpenAI's API, the agent uses **function calling** (tool use). Here's how:

```python
# 1. Define tools in OpenAI's format
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": "Look up an order by ID...",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "..."}
                },
                "required": ["order_id"]
            }
        }
    }
]

# 2. Send messages + tools to the model
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=conversation_history,
    tools=tools,
    tool_choice="auto",  # Let the model decide
)

# 3. If the model wants to call a tool:
if response.choices[0].message.tool_calls:
    tool_call = response.choices[0].message.tool_calls[0]
    result = call_tool(tool_call.function.name, 
                       json.loads(tool_call.function.arguments))
    
    # 4. Feed the result back and get the final response
    messages.append({"role": "tool", "content": json.dumps(result)})
    final_response = client.chat.completions.create(...)
```

The key: **the model decides which tool to call**, but **the tool enforces the rules**.

---

## Part 7: Measuring Agent Performance 📊

In production, you'd track these KPIs:

In [ ]:
# Simulated performance metrics for our agent

metrics = {
    "Total conversations": 1000,
    "Resolved without escalation": 870,
    "Escalated to human": 130,
    "Resolution rate": "87.0%",
    "Avg. handle time": "45 seconds",
    "Cost per ticket (AI)": "$0.85",
    "Cost per ticket (human)": "$12.50",
    "Monthly savings (10.5k AI tickets)": "$122,325",
    "Customer satisfaction": "91%",
}

print("📊 Agent Performance Dashboard")
print("=" * 50)
for key, value in metrics.items():
    print(f"  {key:.<40} {value}")

print("\n💡 Key Takeaways:")
print("  • 87% of tickets resolved without human intervention")
print("  • 93% cost reduction per AI-handled ticket ($12.50 → $0.85)")
print("  • Customer satisfaction improved from 72% to 91%")

---

## Part 8: Next Steps — Going to Production 🚀

This tutorial shows the **core agent pattern**. To make it production-ready, you'd add:

### Architecture Improvements
1. **Real database** — Replace `mock_data.py` with PostgreSQL/MongoDB queries
2. **API layer** — Wrap the agent in a FastAPI/Flask endpoint
3. **Authentication** — Verify customer identity before showing order data
4. **Logging** — Log all tool calls, reasoning steps, and responses

### Safety & Quality
5. **Confidence scoring** — Only auto-respond when confidence > threshold
6. **Human-in-the-loop** — Route uncertain cases for human review
7. **Evaluation suite** — Test against 500+ labeled conversations
8. **Hallucination guards** — Verify the agent only uses data from tool results

### Advanced Features
9. **Multi-turn memory** — Persist conversations across sessions
10. **Proactive notifications** — Alert customers about delays automatically
11. **Multi-language support** — Handle queries in multiple languages
12. **Analytics dashboard** — Real-time monitoring of resolution rates, CSAT, costs

---

## Summary

| Concept | Implementation |
|---|---|
| **Tools** | 4 functions with structured I/O and business rules |
| **Agent Loop** | ReAct: Think → Act → Observe → Respond |
| **Business Rules** | Enforced in tools, NOT the LLM |
| **Fallback** | Mock LLM for running without API key |
| **ROI** | 60% cost reduction, 10x faster response |

**Happy building!** 🛠️